# NCDD Khmer Law OCR — Gemini API

This notebook splits each PDF into 9-page chunks, sends each chunk to Gemini, saves plain-text output, and combines chunks into one `.txt` file per law.

It does not create articles, JSON, or CSV yet. Completed chunks are skipped so the process can resume safely.

In [16]:
%pip install -U google-genai PyMuPDF python-dotenv tqdm


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import csv
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import fitz
from dotenv import load_dotenv
from google import genai
from tqdm.auto import tqdm

load_dotenv()
if not os.getenv('GEMINI_API_KEY'):
    raise RuntimeError('GEMINI_API_KEY is missing. Add it to .env or your environment.')

INPUT_DIR = Path('ncdd_laws/pdfs')
OUTPUT_DIR = Path('ncdd_ocr_gemini_api')
CHUNKS_DIR = OUTPUT_DIR / 'chunks'
CHUNK_TEXT_DIR = OUTPUT_DIR / 'chunk_text'
USAGE_DIR = OUTPUT_DIR / 'usage'
TEXT_DIR = OUTPUT_DIR / 'text'
USAGE_LOG = OUTPUT_DIR / 'usage.log'
USAGE_FIELDS = [
    'timestamp_utc', 'status', 'law_id', 'chunk', 'pdf_path',
    'text_path', 'usage_path', 'attempt', 'model', 'input_tokens',
    'output_tokens', 'thinking_tokens', 'total_tokens'
    'duration_seconds'
]
PAGES_PER_CHUNK = 9
MODEL = 'gemini-3.5-flash'
MAX_RETRIES = 4
COOLDOWN_SECONDS = 30

for folder in (CHUNKS_DIR, CHUNK_TEXT_DIR, USAGE_DIR, TEXT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
print(f'Model: {MODEL}')
print(f'Input: {INPUT_DIR}')

Model: gemini-3.5-flash
Input: ncdd_laws/pdfs


In [18]:
PROMPT = '''# Khmer Law Plain-Text Extraction

Read the uploaded Khmer legal document chunk carefully and extract all available text in its original reading order.

Return only the extracted plain text. Do not return JSON, CSV, Markdown code fences, explanations, summaries, translations, or extraction metadata.

Rules:
1. Preserve the original Khmer text, spelling, punctuation, numbering, and symbols as accurately as possible.
2. Do not translate, summarize, rewrite, correct, or omit any text.
3. Preserve article headings such as មាត្រា១, មាត្រា ១, មាត្រាទី១, មាត្រាទី ១, and Article 1.
4. If a chunk starts with ordinary text without an article heading, preserve it exactly. Do not invent a heading.
5. If an article continues from an earlier page or chunk, preserve the continuation as part of the same article. Do not add a new heading.
6. Preserve meaningful paragraph breaks, but remove only unnecessary line breaks caused by PDF formatting.
7. Preserve all available text, including incomplete text at the beginning or end of this chunk.
8. The example labels below are instructions only, not document content. Never output labels such as Chunk 1 or Chunk 2 unless they actually appear in the document.

Example of consecutive chunks:

Chunk 1 — Article starts normally

មាត្រា១

ច្បាប់នេះមានគោលបំណងកំណត់អំពីសិទ្ធិ និងកាតព្វកិច្ចរបស់បុគ្គល។

មាត្រា២

បុគ្គលគ្រប់រូបត្រូវគោរពតាមបទប្បញ្ញត្តិនៃច្បាប់នេះ។

Chunk 2 — Continuation without article heading

ការអនុវត្តសិទ្ធិទាំងនេះត្រូវធ្វើឡើងដោយគោរពតាមច្បាប់ជាធរមាន។

មាត្រា៣

បុគ្គលគ្រប់រូបមានសិទ្ធិទទួលបានការការពារតាមច្បាប់។

When processing Chunk 2, the text before មាត្រា៣ remains part of មាត្រា២. Do not add a new heading to that continuation. The example labels and example text are instructions only, not document content.

Output only the extracted text from the uploaded document chunk.'''

In [19]:
def split_pdf(pdf_path, pages_per_chunk=PAGES_PER_CHUNK):
    pdf_path = Path(pdf_path)
    law_id = pdf_path.stem
    law_dir = CHUNKS_DIR / law_id
    law_dir.mkdir(parents=True, exist_ok=True)
    chunk_paths = []

    with fitz.open(pdf_path) as source:
        for start in range(0, source.page_count, pages_per_chunk):
            end = min(start + pages_per_chunk, source.page_count)
            number = start // pages_per_chunk + 1
            chunk_path = law_dir / f'{law_id}-{number}.pdf'
            if not chunk_path.exists():
                chunk = fitz.open()
                chunk.insert_pdf(source, from_page=start, to_page=end - 1)
                chunk.save(chunk_path)
                chunk.close()
            chunk_paths.append(chunk_path)
    return chunk_paths

def extract_chunk(chunk_path):
    uploaded = client.files.upload(file=str(chunk_path))
    response = client.models.generate_content(
        model=MODEL,
        contents=[uploaded, PROMPT],
    )
    text = (response.text or '').strip()
    if not text:
        raise RuntimeError(f'Gemini returned empty text for {chunk_path}')
    usage = response.usage_metadata
    token_usage = {
        'model': MODEL,
        'input_tokens': getattr(usage, 'prompt_token_count', None),
        'output_tokens': getattr(usage, 'candidates_token_count', None),
        'thinking_tokens': getattr(usage, 'thoughts_token_count', None),
        'total_tokens': getattr(usage, 'total_token_count', None),
    }
    return text, token_usage

def process_law(pdf_path):
    pdf_path = Path(pdf_path)
    law_id = pdf_path.stem
    chunk_paths = split_pdf(pdf_path)
    law_text_dir = CHUNK_TEXT_DIR / law_id
    law_usage_dir = USAGE_DIR / law_id
    law_text_dir.mkdir(parents=True, exist_ok=True)
    law_usage_dir.mkdir(parents=True, exist_ok=True)

    for index, chunk_path in enumerate(tqdm(chunk_paths, desc=f'Gemini {law_id}')):
        text_path = law_text_dir / f'{chunk_path.stem}.txt'
        usage_path = law_usage_dir / f'{chunk_path.stem}.json'
        if text_path.exists():
            continue
        for attempt in range(MAX_RETRIES):
            try:
                chunk_started = time.perf_counter()
                text, token_usage = extract_chunk(chunk_path)
                duration_seconds = round(time.perf_counter() - chunk_started, 2)
                text_path.write_text(text + '\n', encoding='utf-8')
                usage_path.write_text(json.dumps({
                    'law_id': law_id,
                    'chunk': chunk_path.stem,
                    **token_usage,
                }, ensure_ascii=False, indent=2), encoding='utf-8')
                log_record = {
                    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
                    'status': 'success',
                    'law_id': law_id,
                    'chunk': chunk_path.stem,
                    'pdf_path': str(chunk_path),
                    'text_path': str(text_path),
                    'usage_path': str(usage_path),
                    'attempt': attempt + 1,
                    'duration_seconds': duration_seconds,
                    **token_usage,
                }
                log_exists = USAGE_LOG.exists() and USAGE_LOG.stat().st_size > 0
                with USAGE_LOG.open('a', encoding='utf-8', newline='') as log_file:
                    writer = csv.DictWriter(log_file, fieldnames=USAGE_FIELDS)
                    if not log_exists:
                        writer.writeheader()
                    writer.writerow(log_record)
                print(f"{chunk_path.name}: {token_usage['total_tokens']} total tokens")
                if index < len(chunk_paths) - 1:
                    print(f'Cooling down for {COOLDOWN_SECONDS} seconds...')
                    time.sleep(COOLDOWN_SECONDS)
                break
            except Exception:
                if attempt == MAX_RETRIES - 1:
                    raise
                time.sleep(2 ** attempt)

    final_path = TEXT_DIR / f'{law_id}.txt'
    parts = [
        (CHUNK_TEXT_DIR / law_id / f'{p.stem}.txt').read_text(encoding='utf-8').strip()
        for p in chunk_paths
    ]
    final_path.write_text('\n\n'.join(parts) + '\n', encoding='utf-8')
    return final_path

In [20]:
# Test one law first
result = process_law(INPUT_DIR / "16780.pdf")
print(f'Created: {result}')

# After the test works, process all laws:
# for pdf_path in sorted(INPUT_DIR.glob('*.pdf')):
#     print(process_law(pdf_path))

Gemini 16780:   0%|          | 0/3 [00:00<?, ?it/s]

16780-1.pdf: 19861 total tokens
Cooling down for 30 seconds...


Gemini 16780:  33%|███▎      | 1/3 [01:37<03:14, 97.31s/it]

16780-2.pdf: 21486 total tokens
Cooling down for 30 seconds...


Gemini 16780: 100%|██████████| 3/3 [04:06<00:00, 82.11s/it]

16780-3.pdf: 14386 total tokens
Created: ncdd_ocr_gemini_api/text/16780.txt
